In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import glob
import os
import numpy as np
import pandas as pd

# Search for the uploaded file in Google Drive
matches = glob.glob("/content/drive/MyDrive/**/news_raw_basketA_text.csv", recursive=True)

if not matches:
    raise FileNotFoundError(
        "Could not find 'news_raw_basketA.csv' in Google Drive. "
        "Please check if Google Drive is mounted or if the file name/path is correct."
    )

SRC = matches[0]
OUT = SRC.replace("news_raw_basketA.csv", "news_scores_basketA.csv")
print("reading:", SRC)

In [ ]:
df = pd.read_csv(SRC, low_memory=False); df["row_id"] = df.index

In [ ]:
for name in ["out", "df", "scores", "res"]:
    if name in globals():
        print(name, globals()[name].shape, globals()[name].columns.tolist())

In [ ]:
import glob, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(dev).eval()
id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
pos_i = [k for k, v in id2label.items() if v == "positive"][0]
neg_i = [k for k, v in id2label.items() if v == "negative"][0]

SRC = glob.glob("/content/drive/MyDrive/**/news_raw_basketA_text.csv", recursive=True)[0]
df = pd.read_csv(SRC, low_memory=False)
df["row_id"] = df.index
art = df["Article"].fillna("").astype(str).str.strip()
usable = art.str.len() >= 200
text = (df["Article_title"].fillna("").astype(str) + ". " + art)[usable]
print(len(df), "rows |", int(usable.sum()), f"usable ({usable.mean():.1%})")

order = text.str.len().sort_values().index          # similar lengths together means less padding
scores = pd.Series(np.nan, index=df.index)
BATCH = 32
with torch.no_grad():
    for b, i in enumerate(range(0, len(order), BATCH)):
        idx = order[i:i + BATCH]
        enc = tok(text[idx].tolist(), padding=True, truncation=True, max_length=512,
                  return_tensors="pt").to(dev)
        with torch.autocast(device_type=dev, enabled=(dev == "cuda")):
            logits = model(**enc).logits
        p = torch.softmax(logits.float(), dim=1).cpu().numpy()
        scores.loc[idx] = p[:, pos_i] - p[:, neg_i]
        if b % 100 == 0:
            print(f"{i:,} / {len(order):,}", flush=True)

out = pd.DataFrame({"row_id": df["row_id"], "Date": df["Date"], "Stock_symbol": df["Stock_symbol"],
                    "text_score": scores, "used_text": usable.astype(int)})
out.to_csv("/content/drive/MyDrive/us_experiment/news_scores_basketA_text.csv", index=False)
print("Finished!", out.shape)

In [ ]:
lens = [len(tok(t, truncation=False)["input_ids"]) for t in text.sample(500, random_state=1)]
print("tokens per row (10th, 50th, 90th percentile):", np.percentile(lens, [10, 50, 90]))

h = pd.read_csv("/content/drive/MyDrive/us_experiment/news_scores_basketA.csv")[["row_id", "sentiment_score"]]
m = out.merge(h, on="row_id")
m = m[m["used_text"] == 1]
print("correlation, headline score vs text score:", round(m["text_score"].corr(m["sentiment_score"]), 3))
print("average absolute difference:", round((m["text_score"] - m["sentiment_score"]).abs().mean(), 3))